<a href="https://colab.research.google.com/github/Ruthuja-Gaikwad/DAUP/blob/main/APEX_MOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apex-MOM: Agentic Operational Workflow & Predictive Maintenance Engine
### A Multi-Agent LangGraph Pipeline for Autonomous Smart Factory Incident Response

In [39]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [55]:
print("Listing all available Generative AI models and filtering for generateContent...")

available_models = []
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        available_models.append(m.name)

if available_models:
    print("✅ Found supported generation models:")
    for model_name in available_models:
        print(f"- {model_name}")
else:
    print("❌ No models found that support 'generateContent'. This might indicate a problem with your API key, region, or environment.")

Listing all available Generative AI models and filtering for generateContent...
✅ Found supported generation models:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.5-flash
- models/lyria-3-clip-preview

Now that the API key is configured, let's import the other necessary libraries and initialize the Gemini model.

In [40]:
import os
from typing import List, Literal, Tuple, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.runnables import RunnableLambda

# Initialize the Gemini Pro model
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0, google_api_key=GOOGLE_API_KEY)

# Initialize the embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GOOGLE_API_KEY)

### Set up the Chroma vector store

To demonstrate the RAG chain, we'll create a simple Chroma vector store with some sample documents. In a real application, you would load your own data here.

Now that we have a retriever, we can define the RAG chain. This chain will take a user's question, retrieve relevant documents from the vector store, and then pass both the question and the documents to the LLM to generate an answer.

In [61]:
import os
import shutil
import time
from typing import List, Literal, Tuple, TypedDict
from google.colab import userdata

# 1. Securely set API Key in environment so all LangChain tools can find it
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 2. Initialize the Gemini Model (1.5 Flash is perfect for speed/routing)
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

# 3. Initialize Embeddings (Corrected model string name)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 4. Define Vegam Smart Factory Suite (SFS) Digital SOP Manuals (re-defining for this cell's context)
vegam_factory_sops = [
    "REG-SOP-VIB-102: If a rotating mixer motor or pump registers vibration levels exceeding 4.5 mm/s, it indicates critical Bearing Degradation. Action Required: Immediately throttle rotational speed below 1200 RPM, reduce line feed rate by 50%, and dispatch a Critical Mechanical Requisition.",
    "REG-SOP-TEMP-404: In chemical reactor vessels or heat exchangers, if the process temperature crosses 310 Kelvin while tool wear exceeds 180 minutes, a Heat Dissipation Failure is active. Action Required: Trigger auxiliary coolant flush valves immediately and schedule an urgent maintenance overhaul within 12 hours.",
    "COMPLIANCE-MOM-99: Under factory regulatory frameworks, every generated SAP asset work order must record an authorized MTBF (Mean Time Between Failures) safety rating code and explicitly reference the local equipment physical tag identifier."
]

# 5. Create Chroma entirely in RAM (No file paths = No read-only bugs!)
vector_store = Chroma.from_texts(
    texts=vegam_factory_sops,
    embedding=embeddings
)

# Safe connection handshake to build the retriever component
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

# 6. Define your LangGraph State
class AgentState(TypedDict):
    messages: List[BaseMessage]
    next_step: str
    retrieved_context: str | None

print("✅ LLM, Fixed Embeddings Model, and In-Memory Chroma DB Initialized successfully!")

✅ LLM, Fixed Embeddings Model, and In-Memory Chroma DB Initialized successfully!


In [46]:
# 1. Install graph visualization helpers & clean up remaining dependencies
!pip install -q grandalf matplotlib

# 2. Fix the OpenTelemetry version conflicts so Google Colab & LangChain play nice
!pip install -q "opentelemetry-api==1.38.0" "opentelemetry-sdk==1.38.0" "opentelemetry-proto==1.38.0" "opentelemetry-exporter-otlp-proto-common==1.38.0"

### Fixing the ChromaDB 'Read-Only' Error with In-Memory Persistence

In [56]:
import os
import gc
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. Enforce strict environment variables
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# 2. Force Python to release any lingering database connections in the background
gc.collect()

# 3. Initialize the Gemini Pro Brain
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

# 4. Initialize your verified Gemini Embedding Engine
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 5. Define Vegam Smart Factory Suite (SFS) Digital SOP Manuals
vegam_factory_sops = [
    "REG-SOP-VIB-102: If a rotating mixer motor or pump registers vibration levels exceeding 4.5 mm/s, it indicates critical Bearing Degradation. Action Required: Immediately throttle rotational speed below 1200 RPM, reduce line feed rate by 50%, and dispatch a Critical Mechanical Requisition.",
    "REG-SOP-TEMP-404: In chemical reactor vessels or heat exchangers, if the process temperature crosses 310 Kelvin while tool wear exceeds 180 minutes, a Heat Dissipation Failure is active. Action Required: Trigger auxiliary coolant flush valves immediately and schedule an urgent maintenance overhaul within 12 hours.",
    "COMPLIANCE-MOM-99: Under factory regulatory frameworks, every generated SAP asset work order must record an authorized MTBF (Mean Time Between Failures) safety rating code and explicitly reference the local equipment physical tag identifier."
]

# 6. FIX: Create Chroma entirely in RAM (No file paths = No read-only bugs!)
vector_store = Chroma.from_texts(
    texts=vegam_factory_sops,
    embedding=embeddings
)

# 7. Safe connection handshake to build the retriever component
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

print("🚀 Success! In-Memory Chroma DB built and populated without file system locks.")

🚀 Success! In-Memory Chroma DB built and populated without file system locks.


### Verify In-Memory ChromaDB

In [50]:
# Create the prompt that guides our factory coordinator
router_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert AI Factory Operations Coordinator for the Vegam Smart Factory Suite (SFS).\n\n"
        "Your task is to analyze incoming user queries or factory alerts and decide on the next step:\n"
        "1. If the message mentions specific machine alerts, sensor thresholds (vibration, temperature, RPM, Kelvin), "
        "or compliance codes, output exactly: 'RETRIEVE'\n"
        "2. If it is general greeting, a casual question, or something that doesn't need factory manual lookups, "
        "output exactly: 'RESPOND'\n\n"
        "Do not include any other text, punctuation, or explanation. Output exactly one word: 'RETRIEVE' or 'RESPOND'."
    )),
    MessagesPlaceholder(variable_name="messages"),
])

print("🤖 Router prompt template successfully defined.")

🤖 Router prompt template successfully defined.


In [51]:
from langchain_core.messages import AIMessage

# --- NODE 1: The Router Node ---
def router_node(state: AgentState):
    """Analyzes the message and decides whether to retrieve information or respond directly."""
    print("🧠 [Node: Router] Analyzing incoming request...")
    messages = state["messages"]

    # Format prompt with conversation history
    formatted_prompt = router_prompt.format_messages(messages=messages)
    decision = llm.invoke(formatted_prompt).content.strip()

    # Update the graph state with the decision
    return {"next_step": decision}


# --- NODE 2: The Retrieval Node ---
def retrieval_node(state: AgentState):
    """Queries Chroma DB to find the exact factory SOP matching the user's issue."""
    print("🔍 [Node: Retriever] Searching Chroma Vector DB for matching SOPs...")
    messages = state["messages"]
    last_user_message = messages[-1].content

    # Fetch the single best match (k=1) from Chroma
    docs = retriever.invoke(last_user_message)
    context = docs[0].page_content if docs else "No specific SOP found for this issue."

    # Build an engineered response prompt injection
    system_instruction = (
        f"You are the Vegam SFS Specialist. Answer the user's question using ONLY the following "
        f"official factory SOP manual rule. If the rule doesn't cover the problem, state that explicitly.\n\n"
        f"OFFICIAL SOP CONTEXT:\n{context}"
    )

    # Combine instructions with the conversation history and ask Gemini
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        MessagesPlaceholder(variable_name="messages")
    ])

    response = llm.invoke(qa_prompt.format_messages(messages=messages))

    # Append the AI's final answer to the message history
    return {"messages": [response]}


# --- NODE 3: The Direct Response Node ---
def direct_response_node(state: AgentState):
    """Handles standard conversations (greetings, chit-chat) without touching the database."""
    print("💬 [Node: Direct Responder] Handling standard conversational chat...")
    messages = state["messages"]

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an exceptionally helpful and friendly AI operations assistant for Vegam Smart Factory Suite. Answer nicely."),
        MessagesPlaceholder(variable_name="messages")
    ])

    response = llm.invoke(chat_prompt.format_messages(messages=messages))
    return {"messages": [response]}

print("⚙️ All three workflow nodes (Router, Retriever, Responder) are compiled and ready.")

⚙️ All three workflow nodes (Router, Retriever, Responder) are compiled and ready.


In [63]:
import json
from langchain_core.messages import AIMessage, HumanMessage # Ensure HumanMessage is imported
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # Ensure MessagesPlaceholder is imported

# --- Node 1: The Routing Decision Node (Stays the same) ---
def factory_router_node(state: AgentState) -> dict:
    print("\n[Node 1: Factory Router] Analyzing incoming alert payload...")
    router_chain = router_prompt | llm
    response = router_chain.invoke({"messages": state["messages"]})
    decision = response.content.strip().upper()
    print(f" -> Decision: Routing to '{decision}' phase.")
    return {
        "next_step": decision,
        "messages": state["messages"] # Ensure messages are passed through
    }

# --- Node 2: The Retriever Node (FIXED: Uses HumanMessage wrapper for context injection) ---
def asset_retriever_node(state: AgentState) -> dict:
    print("[Node 2: Asset Retriever] Extracting context from Vegam SFS Digital Manuals...")

    # Grab the initial telemetry alert message
    user_alert = state["messages"][-1].content

    # Query our In-Memory Chroma instance
    matched_docs = retriever.invoke(user_alert)
    context_string = "\n".join([doc.page_content for doc in matched_docs])

    # FIX: We wrap the RAG facts into a hidden Human/User role message context block.
    # This keeps Gemini's internal turn-taking validator perfectly happy!
    rag_context_message = HumanMessage(
        content=f"[SYSTEM FILE INGESTION - DO NOT ANSWER YET]\nContext Rules:\n{context_string}"
    )

    # Append the RAG context as a HumanMessage to the existing messages
    current_messages = state["messages"]
    current_messages.append(rag_context_message)

    return {
        "messages": current_messages,
        "next_step": "RESPOND"
    }

# --- Node 3: The Responder Node (FIXED: Appends final User request to seal the turn) ---
def operations_responder_node(state: AgentState) -> dict:
    print("[Node 3: Operations Responder] Generating final engineering action directive...")

    # Pull the original alert text out of the very first message in the graph history
    original_alert = state["messages"][0].content

    synthesis_prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are the execution intelligence interface for the Vegam Smart Factory Suite.\n"
            "Review the historical context data injected in the message thread history and output "
            "a comprehensive, professional engineering action directive to address the maintenance alert."
        )),
        MessagesPlaceholder(variable_name="messages"),
        # FIX: Force the absolute final turn item to be a clean User confirmation request
        ("human", "Synthesize the data and generate the final operational directive for: {alert_query}")
    ])

    responder_chain = synthesis_prompt | llm

    # Execute the chain while feeding the query variable explicitly
    final_output = responder_chain.invoke({
        "messages": state["messages"],
        "alert_query": original_alert
    })

    return {
        "messages": [final_output], # Return only the final AI response
        "next_step": "END"
    }

In [67]:
import re

# --- New Node: Invalid Decision Handler ---
def invalid_decision_node(state: AgentState) -> dict:
    print("[Node 4: Invalid Decision Handler] Router could not make a clear decision.")
    # Optionally, you can log the raw output for debugging
    raw_output = state.get("router_raw_output", "No raw output available.")
    decision_message = (
        f"I'm sorry, I couldn't clearly understand your request based on the factory manuals. "
        f"Could you please rephrase or provide more specific details? "
        f"(Router's raw output: {raw_output})"
    )
    return {"messages": [HumanMessage(content=decision_message)]}


In [68]:
from langgraph.graph import StateGraph, END

# Initialize our state-driven graph map using the AgentState definition
workflow = StateGraph(AgentState)

# 1. Register our operational code chunks as graph nodes
workflow.add_node("router", factory_router_node)
workflow.add_node("retriever", asset_retriever_node)
workflow.add_node("responder", operations_responder_node)
workflow.add_node("invalid_decision", invalid_decision_node) # Add the new node here

# 2. Establish our workflow entry point
workflow.set_entry_point("router")

# 3. Design the conditional branching logic rules
def routing_conditional_edge(state: AgentState) -> Literal["retriever", "responder", "invalid_decision"]:
    # Read the data returned by the router node to decide the track path
    if state["next_step"] == "RETRIEVE":
        return "retriever"
    elif state["next_step"] == "RESPOND":
        return "responder"
    else:
        return "invalid_decision"

# Add the structural routes from the router node
workflow.add_conditional_edges(
    "router",
    routing_conditional_edge,
    {
        "retriever": "retriever",
        "responder": "responder",
        "invalid_decision": "invalid_decision" # Add the new route here
    }
)

# 4. Connect the remaining fixed paths to the final state boundaries
workflow.add_edge("retriever", "responder")
workflow.add_edge("responder", END)
workflow.add_edge("invalid_decision", END) # Add edge from invalid_decision to END

# 5. Compile our complete factory graph application
vegam_agent_engine = workflow.compile()
print("⚙️ LangGraph Workflow Engine completely assembled, mapped, and compiled successfully!")

⚙️ LangGraph Workflow Engine completely assembled, mapped, and compiled successfully!


In [69]:
from langchain_core.messages import HumanMessage

factory_incident_payload = "What is the capital of France?"

print(f"Initiating Live Factory Simulation Run...\nInput Alert: {factory_incident_payload}")

initial_graph_state = {
    "messages": [HumanMessage(content=factory_incident_payload)]
}

final_state_output = vegam_agent_engine.invoke(initial_graph_state)

print("\n================🏭 AGENT FINAL ENGINEERING DIRECTIVE ================")
print(final_state_output["messages"][-1].content)
print("=====================================================================")

Initiating Live Factory Simulation Run...
Input Alert: What is the capital of France?

[Node 1: Factory Router] Analyzing incoming alert payload...
 -> Decision: Routing to 'RESPOND' phase.
[Node 3: Operations Responder] Generating final engineering action directive...

================🏭 AGENT FINAL ENGINEERING DIRECTIVE ================
As the Vegam Smart Factory Suite execution intelligence interface, I have processed your input.

Regarding your direct question: The capital of France is **Paris**.

However, the request to "Synthesize the data and generate the final operational directive for: What is the capital of France?" cannot be fulfilled in the context of an engineering action directive for a maintenance alert. The input "What is the capital of France?" is a general knowledge query and does not represent a maintenance alert, operational anomaly, or data point requiring an engineering intervention or a directive from the Vegam Smart Factory Suite.

To generate a comprehensive, pr

In [70]:
from langchain_core.messages import HumanMessage

factory_incident_payload = "CRITICAL: Mixer asset MX-402 is experiencing massive bearing movement. Vibration readings are spiking at 4.92 mm/s."

print(f"Initiating Live Factory Simulation Run...\nInput Alert: {factory_incident_payload}")

initial_graph_state = {
    "messages": [HumanMessage(content=factory_incident_payload)]
}

final_state_output = vegam_agent_engine.invoke(initial_graph_state)

print("\n================🏭 AGENT FINAL ENGINEERING DIRECTIVE ================")
print(final_state_output["messages"][-1].content)
print("=====================================================================")

Initiating Live Factory Simulation Run...
Input Alert: CRITICAL: Mixer asset MX-402 is experiencing massive bearing movement. Vibration readings are spiking at 4.92 mm/s.

[Node 1: Factory Router] Analyzing incoming alert payload...
 -> Decision: Routing to 'RETRIEVE' phase.
[Node 2: Asset Retriever] Extracting context from Vegam SFS Digital Manuals...
[Node 3: Operations Responder] Generating final engineering action directive...

================🏭 AGENT FINAL ENGINEERING DIRECTIVE ================
**ENGINEERING ACTION DIRECTIVE**

**Subject:** CRITICAL MAINTENANCE ALERT - Mixer Asset MX-402 - Bearing Degradation

**Urgency:** CRITICAL

**Date:** [Current Date]
**Time:** [Current Time]

**Alert Origin:** Vegam Smart Factory Suite - Vibration Monitoring System

**Asset ID:** MX-402
**Condition:** Critical Bearing Degradation detected. Vibration readings are spiking at 4.92 mm/s, indicating massive bearing movement.

**Governing Standard:** REG-SOP-VIB-102 - Critical Bearing Degradation